In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_csv("crop_yield_final_cleaned.csv")

X = df.drop(columns=["Yield_tons_per_hectare"])
y = df["Yield_tons_per_hectare"]


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [6]:
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


In [7]:
from xgboost import XGBRegressor


xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42
)

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
results = {}
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

mae, rmse, r2 = evaluate_model(y_test, y_pred_xgb)

results["XGBoost"] = {
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
}


In [15]:
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)
results_df


,MAE,RMSE,R2
XGBoost,0.3988,0.4999,0.913


In [ ]:
import joblib

joblib.dump(xgb_model, "xgboost.pkl")